# Extracting Variable and Value Labels

We will need to know what the numbers mean to know what to do next for processing or analysis.
We will extract this information from the SPSS syntax files.
There are two procedures of interest:
1. `VARIABLE LABEL`: This describes what the variable measures or describes.
2. `VALUE LABEL`: This describes what each code value used in that variable means.

In [1]:
import pandas as pd
import re

# 0. List of Syntax Files

The syntax file request result metadata conveniently also serves as a reminder of the names that the files have been stored as.

In [2]:
syntax_files = pd.read_csv('data/request_result_syntax.csv', index_col=[0])
syntax_files

,persistentId,description,num_fails
syntax_1999-nov-dec_rebase2001.txt,hdl:11272.1/AB2/LGGEKU/FI1VWQ,SPSS syntax November-December 1999,0
syntax_1999-jan-oct_rebase2001.txt,hdl:11272.1/AB2/LGGEKU/FFZGU0,SPSS syntax January-October 1999,0
syntax_2000-2010_rebase2001.txt,hdl:11272.1/AB2/F0G05L/DKBI2F,SPSS syntax 2000-2010,0
syntax_1987-1998_rebase2001.txt,hdl:11272.1/AB2/PW64LJ/MVVTLK,SPSS syntax 1987-1998,0
syntax_1976-1986_rebase2001.txt,hdl:11272.1/AB2/SS4B9S/RU28Z5,SPSS syntax 1971-1986,0
syntax_2022_rebase2023.txt,hdl:11272.1/AB2/SRAU7E/S0JZL7,SPSS syntax for plain-text microdata files. Cr...,0
syntax_2021_rebase2023.txt,hdl:11272.1/AB2/HP9TEK/7UJJRS,SPSS syntax for plain-text microdata files. Cr...,0
syntax_2020_rebase2023.txt,hdl:11272.1/AB2/GGXMM2/X7TBKH,SPSS syntax for plain-text microdata files. Cr...,0
syntax_2019_rebase2023.txt,hdl:11272.1/AB2/ATGWRX/K75CHT,SPSS syntax for plain-text microdata files. Cr...,0
syntax_2018_rebase2023.txt,hdl:11272.1/AB2/CFRSC0/GSGTYS,SPSS syntax for plain-text microdata files. Cr...,0


We will process the files in reverse chronological order: from most recent to oldest, from latest rebase version to last.

In [3]:
syntax_files = syntax_files.index.tolist()
syntax_files = sorted(syntax_files, key=lambda f: f.split('_')[1], reverse=True)
print(syntax_files)

['syntax_2022_rebase2023.txt', 'syntax_2021_rebase2023.txt', 'syntax_2020_rebase2023.txt', 'syntax_2019_rebase2023.txt', 'syntax_2018_rebase2023.txt', 'syntax_2017_rebase2023.txt', 'syntax_2016_rebase2023.txt', 'syntax_2015_rebase2023.txt', 'syntax_2014_rebase2023.txt', 'syntax_2013_rebase2023.txt', 'syntax_2012_rebase2023.txt', 'syntax_2011_rebase2023.txt', 'syntax_2010_rebase2023.txt', 'syntax_2009_rebase2023.txt', 'syntax_2008_rebase2023.txt', 'syntax_2007_rebase2023.txt', 'syntax_2006_rebase2023.txt', 'syntax_2000-2010_rebase2001.txt', 'syntax_1999-nov-dec_rebase2001.txt', 'syntax_1999-jan-oct_rebase2001.txt', 'syntax_1987-1998_rebase2001.txt', 'syntax_1976-1986_rebase2001.txt']


# 1. Test Run Extraction Procedure

We will show using one example how the procedure works.
First we define all the regex patterns that will be used to extract the needed information.

In [4]:
proc_re = {
    'varlab': re.compile(r'variable\s+label(?:"[^"]+"|\'[^\']+\'|[^\.])+?\.', flags=re.I),
    'varlab_var': re.compile(r'/?\W*([a-zA-Z0-9_]+)\s*(?:"(.+?)"|\'(.+?)\')', flags=re.I),
    'vallab': re.compile(r'value\s+label(?:".+?"|\'.+?\'|[^\.])+?\.', flags=re.I),
    'vallab_var': re.compile(r'/\W*([a-zA-Z0-9_]+)((?:".+?"|\'.+?\'|[^/])+)', flags=re.I),
    'vallab_lab': re.compile(r'(?:([0-9]+)\s*(?i:"(.+?)"|\'(.+?)\'))', flags=re.I)
}

We load a syntax file and show an example printout of what SPSS syntax code looks like.

In [5]:
syntax = str()
with open('data/{}'.format(syntax_files[0]), 'rt', encoding='utf-8-sig') as f:
    syntax = f.read()
print(syntax[:200])

SET UNICODE YES.

COMMENT Formatted as per Statcan PRN.

DATA LIST FILE="pubMM22.prn" 
/REC_NUM 1-7
SURVYEAR 8-11
SURVMNTH 12-13
LFSSTAT 14-14
PROV 15-16
CMA 17-17
AGE_12 18-19
AGE_6 20-20
SEX 21-21
M


We extract the `VARIABLE LABEL` procedure from the loaded syntax file and show an example printout of this command.

In [6]:
varlab_syntax = proc_re['varlab'].search(syntax).group(0)
print(varlab_syntax)

VARIABLE LABELS
REC_NUM 'Order of record in file'
/SURVYEAR 'Survey year'
/SURVMNTH 'Survey month'
/LFSSTAT 'Labour force status'
/PROV 'Province'
/CMA 'Nine largest CMAs'
/AGE_12 'Five-year age group of respondent'
/AGE_6 'Age in 2 and 3 year groups, 15 to 29'
/SEX 'Sex of respondent'
/MARSTAT 'Marital status of respondent'
/EDUC 'Highest educational attainment'
/MJH 'Single or multiple jobholder'
/EVERWORK 'Not currently employed, worked in the past'
/FTPTLAST 'Full- or part-time status of last job'
/COWMAIN 'Class of worker, main job'
/IMMIG 'Immigrant status'
/NAICS_21 'Industry of main job'
/NOC_10 'Occupation at main job'
/NOC_43 'Occupation at main job'
/YABSENT 'Reason of absence, full week'
/WKSAWAY 'Number of weeks absent from work'
/PAYAWAY 'Paid for time off, full-week absence only'
/UHRSMAIN 'Usual hours worked per week at main job'
/AHRSMAIN 'Actual hours worked per week at main job'
/FTPTMAIN 'Full- or part-time status at main or only job'
/UTOTHRS 'Usual hours worked pe

We extract the variable names and labels into a dictionary and show an example of what the final result looks like.

In [7]:
varlab_syntax = proc_re['varlab_var'].findall(varlab_syntax)
varlab_syntax = {v: a if a else b for v, a, b in varlab_syntax}
varlab_syntax

{'REC_NUM': 'Order of record in file',
 'SURVYEAR': 'Survey year',
 'SURVMNTH': 'Survey month',
 'LFSSTAT': 'Labour force status',
 'PROV': 'Province',
 'CMA': 'Nine largest CMAs',
 'AGE_12': 'Five-year age group of respondent',
 'AGE_6': 'Age in 2 and 3 year groups, 15 to 29',
 'SEX': 'Sex of respondent',
 'MARSTAT': 'Marital status of respondent',
 'EDUC': 'Highest educational attainment',
 'MJH': 'Single or multiple jobholder',
 'EVERWORK': 'Not currently employed, worked in the past',
 'FTPTLAST': 'Full- or part-time status of last job',
 'COWMAIN': 'Class of worker, main job',
 'IMMIG': 'Immigrant status',
 'NAICS_21': 'Industry of main job',
 'NOC_10': 'Occupation at main job',
 'NOC_43': 'Occupation at main job',
 'YABSENT': 'Reason of absence, full week',
 'WKSAWAY': 'Number of weeks absent from work',
 'PAYAWAY': 'Paid for time off, full-week absence only',
 'UHRSMAIN': 'Usual hours worked per week at main job',
 'AHRSMAIN': 'Actual hours worked per week at main job',
 'FTPTMA

We extract the `VALUE LABEL` procedure and show an example printout of what this command looks like.

In [8]:
vallab_syntax = proc_re['vallab'].search(syntax).group(0)
print(vallab_syntax)

VALUE LABELS
REC_NUM
/SURVYEAR
/SURVMNTH
 1 "January"
 2 "February"
 3 "March"
 4 "April"
 5 "May"
 6 "June"
 7 "July"
 8 "August"
 9 "September"
 10 "October"
 11 "November"
 12 "December"
/ LFSSTAT    
 1 "Employed, at work"
 2 "Employed, absent from work"
 3 "Unemployed"
 4 "Not in labour force"
/PROV
 10 "Newfoundland and Labrador"
 11 "Prince Edward Island"
 12 "Nova Scotia"
 13 "New Brunswick"
 24 "Quebec"
 35 "Ontario"
 46 "Manitoba"
 47 "Saskatchewan"
 48 "Alberta"
 59 "British Columbia"
/CMA
 1 "Québec"
 2 "Montréal"
 3 "Ottawa–Gatineau (Ontario part)"
 4 "Toronto"
 5 "Hamilton"
 6 "Winnipeg"
 7 "Calgary"
 8 "Edmonton"
 9 "Vancouver"
 0 "Other CMA or non-CMA"
/AGE_12
 01 "15 to 19 years"
 02 "20 to 24 years"
 03 "25 to 29 years"
 04 "30 to 34 years"
 05 "35 to 39 years"
 06 "40 to 44 years"
 07 "45 to 49 years"
 08 "50 to 54 years"
 09 "55 to 59 years"
 10 "60 to 64 years"
 11 "65 to 69 years"
 12 "70 and over"
/AGE_6    
 1 "15 to 16 years"
 2 "17 to 19 years"
 3 "20 to 21 ye

We will proceed with information extraction in two steps.
First we extract the variable names and value labels as a single string and show what the output looks like.

In [9]:
vallab_syntax = proc_re['vallab_var'].findall(vallab_syntax)
vallab_syntax = {v: l for v, l in vallab_syntax}
vallab_syntax

{'SURVYEAR': '\n',
 'SURVMNTH': '\n 1 "January"\n 2 "February"\n 3 "March"\n 4 "April"\n 5 "May"\n 6 "June"\n 7 "July"\n 8 "August"\n 9 "September"\n 10 "October"\n 11 "November"\n 12 "December"\n',
 'LFSSTAT': '    \n 1 "Employed, at work"\n 2 "Employed, absent from work"\n 3 "Unemployed"\n 4 "Not in labour force"\n',
 'PROV': '\n 10 "Newfoundland and Labrador"\n 11 "Prince Edward Island"\n 12 "Nova Scotia"\n 13 "New Brunswick"\n 24 "Quebec"\n 35 "Ontario"\n 46 "Manitoba"\n 47 "Saskatchewan"\n 48 "Alberta"\n 59 "British Columbia"\n',
 'CMA': '\n 1 "Québec"\n 2 "Montréal"\n 3 "Ottawa–Gatineau (Ontario part)"\n 4 "Toronto"\n 5 "Hamilton"\n 6 "Winnipeg"\n 7 "Calgary"\n 8 "Edmonton"\n 9 "Vancouver"\n 0 "Other CMA or non-CMA"\n',
 'AGE_12': '\n 01 "15 to 19 years"\n 02 "20 to 24 years"\n 03 "25 to 29 years"\n 04 "30 to 34 years"\n 05 "35 to 39 years"\n 06 "40 to 44 years"\n 07 "45 to 49 years"\n 08 "50 to 54 years"\n 09 "55 to 59 years"\n 10 "60 to 64 years"\n 11 "65 to 69 years"\n 12 "70 

Then from each string we extract the codes and code descriptions into a dictionary and show a printout of the final result.

In [10]:
vallab_syntax = {v: {int(n): a if a else b for n, a, b, in proc_re['vallab_lab'].findall(l)} for v, l in vallab_syntax.items()}
vallab_syntax

{'SURVYEAR': {},
 'SURVMNTH': {1: 'January',
  2: 'February',
  3: 'March',
  4: 'April',
  5: 'May',
  6: 'June',
  7: 'July',
  8: 'August',
  9: 'September',
  10: 'October',
  11: 'November',
  12: 'December'},
 'LFSSTAT': {1: 'Employed, at work',
  2: 'Employed, absent from work',
  3: 'Unemployed',
  4: 'Not in labour force'},
 'PROV': {10: 'Newfoundland and Labrador',
  11: 'Prince Edward Island',
  12: 'Nova Scotia',
  13: 'New Brunswick',
  24: 'Quebec',
  35: 'Ontario',
  46: 'Manitoba',
  47: 'Saskatchewan',
  48: 'Alberta',
  59: 'British Columbia'},
 'CMA': {1: 'Québec',
  2: 'Montréal',
  3: 'Ottawa–Gatineau (Ontario part)',
  4: 'Toronto',
  5: 'Hamilton',
  6: 'Winnipeg',
  7: 'Calgary',
  8: 'Edmonton',
  9: 'Vancouver',
  0: 'Other CMA or non-CMA'},
 'AGE_12': {1: '15 to 19 years',
  2: '20 to 24 years',
  3: '25 to 29 years',
  4: '30 to 34 years',
  5: '35 to 39 years',
  6: '40 to 44 years',
  7: '45 to 49 years',
  8: '50 to 54 years',
  9: '55 to 59 years',
  10:

# 2. Extract Information in all Syntax Files

We apply the same procedure to all syntax files now.

In [11]:
varlab = dict()
vallab = dict()
for sf in syntax_files:
    with open('data/{}'.format(sf), 'rt', encoding='utf-8-sig') as f:
        syntax = f.read()
    varlab_syntax = proc_re['varlab'].search(syntax).group(0)
    varlab_syntax = proc_re['varlab_var'].findall(varlab_syntax)
    varlab_syntax = {v: a if a else b for v, a, b in varlab_syntax}
    varlab[sf] = varlab_syntax
    vallab_syntax = proc_re['vallab'].search(syntax).group(0)
    vallab_syntax = proc_re['vallab_var'].findall(vallab_syntax)
    vallab_syntax = {v: l for v, l in vallab_syntax}
    vallab_syntax = {v: {int(n): a if a else b for n, a, b, in proc_re['vallab_lab'].findall(l)} for v, l in vallab_syntax.items()}
    vallab[sf] = vallab_syntax

One way to display all information and to see whether codes have changed meaning over time is to display the data in a table.
First are the variable labels.

In [12]:
pd.DataFrame.from_dict(varlab)

,syntax_2022_rebase2023.txt,syntax_2021_rebase2023.txt,syntax_2020_rebase2023.txt,syntax_2019_rebase2023.txt,syntax_2018_rebase2023.txt,syntax_2017_rebase2023.txt,syntax_2016_rebase2023.txt,syntax_2015_rebase2023.txt,syntax_2014_rebase2023.txt,syntax_2013_rebase2023.txt,...,syntax_2010_rebase2023.txt,syntax_2009_rebase2023.txt,syntax_2008_rebase2023.txt,syntax_2007_rebase2023.txt,syntax_2006_rebase2023.txt,syntax_2000-2010_rebase2001.txt,syntax_1999-nov-dec_rebase2001.txt,syntax_1999-jan-oct_rebase2001.txt,syntax_1987-1998_rebase2001.txt,syntax_1976-1986_rebase2001.txt
REC_NUM,Order of record in file,Order of record in file,Order of record in file,Order of record in file,Order of record in file,Order of record in file,Order of record in file,Order of record in file,Order of record in file,Order of record in file,...,Order of record in file,Order of record in file,Order of record in file,Order of record in file,Order of record in file,Order of record in file,Order of record in file,Order of record in file,Order of record in file,Order of record in file
SURVYEAR,Survey year,Survey year,Survey year,Survey year,Survey year,Survey year,Survey year,Survey year,Survey year,Survey year,...,Survey year,Survey year,Survey year,Survey year,Survey year,Survey year,Survey year,Survey year,Survey year,Survey year
SURVMNTH,Survey month,Survey month,Survey month,Survey month,Survey month,Survey month,Survey month,Survey month,Survey month,Survey month,...,Survey month,Survey month,Survey month,Survey month,Survey month,Survey month,Survey month,Survey month,Survey month,Survey month
LFSSTAT,Labour force status,Labour force status,Labour force status,Labour force status,Labour force status,Labour force status,Labour force status,Labour force status,Labour force status,Labour force status,...,Labour force status,Labour force status,Labour force status,Labour force status,Labour force status,Labour force status,Labour force status,Labour force status,Labour force status,Labour force status
PROV,Province,Province,Province,Province,Province,Province,Province,Province,Province,Province,...,Province,Province,Province,Province,Province,Province,Province,Province,Province,Province
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
SP_UHRST,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,"Spouse's usual hours at ALL jobs, employed","Spouse's usual hours at ALL jobs, employed","Spouse's usual hours at ALL jobs, employed","Spouse's usual hours at ALL jobs, employed","Spouse's usual hours at ALL jobs, employed"
SP_COWM,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,"Spouse's class of worker, main job, employed","Spouse's class of worker, main job, employed","Spouse's class of worker, main job, employed","Spouse's class of worker, main job, employed","Spouse's class of worker, main job, employed"
AGYOWNKN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,Age of youngest own child (children),Age of youngest own child (children),Age of youngest own child (children),Age of youngest own child (children),Age of youngest own child (children)
SCH1624,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,At least one child age 16 to 24 in school,At least one child age 16 to 24 in school,At least one child age 16 to 24 in school,At least one child age 16 to 24 in school,At least one child age 16 to 24 in school


Second are the value labels.

In [13]:
pd.concat([pd.DataFrame.from_dict(data, orient='index').stack().rename(file) for file, data in vallab.items()], axis=1)

syntax_2022_rebase2023.txt syntax_2021_rebase2023.txt   
SURVMNTH 1                    January                    January  \
         2                   February                   February   
         3                      March                      March   
         4                      April                      April   
         5                        May                        May   
...                               ...                        ...   
AGYOWNKN 3                        NaN                        NaN   
         4                        NaN                        NaN   
         5                        NaN                        NaN   
         6                        NaN                        NaN   
SCH1624  1                        NaN                        NaN   

           syntax_2020_rebase2023.txt syntax_2019_rebase2023.txt   
SURVMNTH 1                    January                    January  \
         2                   February                   February   
         3                      March                      March   
         4                      April                      April   
         5                        May                        May   
...                               ...                        ...   
AGYOWNKN 3                        NaN                        NaN   
         4                        NaN                        NaN   
         5                        NaN                        NaN   
         6                        NaN                        NaN   
SCH1624  1                        NaN                        NaN   

           syntax_2018_rebase2023.txt syntax_2017_rebase2023.txt   
SURVMNTH 1                    January                    January  \
         2                   February                   February   
         3                      March                      March   
         4                      April                      April   
         5                        May                        May   
...                               ...                        ...   
AGYOWNKN 3                        NaN                        NaN   
         4                        NaN                        NaN   
         5                        NaN                        NaN   
         6                        NaN                        NaN   
SCH1624  1                        NaN                        NaN   

           syntax_2016_rebase2023.txt syntax_2015_rebase2023.txt   
SURVMNTH 1                    January                    January  \
         2                   February                   February   
         3                      March                      March   
         4                      April                      April   
         5                        May                        May   
...                               ...                        ...   
AGYOWNKN 3                        NaN                        NaN   
         4                        NaN                        NaN   
         5                        NaN                        NaN   
         6                        NaN                        NaN   
SCH1624  1                        NaN                        NaN   

           syntax_2014_rebase2023.txt syntax_2013_rebase2023.txt  ...   
SURVMNTH 1                    January                    January  ...  \
         2                   February                   February  ...   
         3                      March                      March  ...   
         4                      April                      April  ...   
         5                        May                        May  ...   
...                               ...                        ...  ...   
AGYOWNKN 3                        NaN                        NaN  ...   
         4                        NaN                        NaN  ...   
         5                        NaN                        NaN  ...   
         6   